# Build Tiled Library

To speed up mapping, we will tile segment-based library where the FSP-FPP relations are organized by tiles. This task includes the following main steps:

* Tile segment-based library
* Calculate FSP and segment downstream distance
* Assign stream order to segments and FSPs

## Import Python Packages

In [1]:
import os

# import the tile module from the fldpln package
from fldpln.tile import *

## Tile Segment-based Library

Here we spatially divide a segment-based library into tiles. Note that tile size is the number of cells to avoid partial cells within a tile and can be used for both PCS and GCS. 

This process also copies the FSP and segment info CSV files to the tiled library and creates a metadata file (TileCellSizeSpatialReference.json) which stores library tile and cell sizes and spatial reference in a JSON file.

**Note that this is the most time consuming process and may take hours for large libraries.**

In [2]:
# Segment-based library
segLibFolder = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/seglib'

# tiled library folder
tiledLibFolder = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/tiledlib' 

# define tile size (number of cells) and format
cellSize = 10
tileSize = 500 # number of cells
tileFileFormat = 'snappy' # 'snappy' or 'mat'

# tile the library
print(f'Tile library: {segLibFolder} ...')
TileLibrary(segLibFolder, cellSize, tiledLibFolder,tileSize,tileFileFormat) 

Tile library: E:/fldpln/workshop/devcon26/wildcat_10m_3dep/seglib ...
Calculate library extent ...
Library external border extent (minX, maxX, minY, maxY) : (np.float64(-64893.2057939356), np.float64(-39593.2057939356), np.float64(1785352.9445651), np.float64(1801822.9445651))
Total number of FSP-FPP relations: 1546815
Number of (possible) tiles: 24
Tile extents:
 [(np.float64(-64893.2057939356), np.float64(-59893.2057939356), np.float64(1785352.9445651), np.float64(1790352.9445651)), (np.float64(-64893.2057939356), np.float64(-59893.2057939356), np.float64(1790352.9445651), np.float64(1795352.9445651)), (np.float64(-64893.2057939356), np.float64(-59893.2057939356), np.float64(1795352.9445651), np.float64(1800352.9445651)), (np.float64(-64893.2057939356), np.float64(-59893.2057939356), np.float64(1800352.9445651), np.float64(1805352.9445651)), (np.float64(-59893.2057939356), np.float64(-54893.2057939356), np.float64(1785352.9445651), np.float64(1790352.9445651)), (np.float64(-59893.205

{'TileSize': 500,
 'CellSize': 10,
 'SpatialReference': 'PROJCS["NAD_1983_Contiguous_USA_Albers",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",29.5],PARAMETER["Standard_Parallel_2",45.5],PARAMETER["Latitude_Of_Origin",23.0],UNIT["Meter",1.0]]\n'}

## Calculate FSP and Segment Downstream Distance

Here we calculate downstream distance for both FSPs and segments for use in inundation mapping. It involves the following major sub-tasks:
* Clean up segments 
    * removes segments from the segment table if they are not in the FSP table. 
    * If a removed segment is the downstream segment of another segment in the segment table, the upstream segment ID is set to 0 (i.e., watershed outlet). 
    * Those removed segments are usually close to or in waterbodies. By removing those segments, a library may have several separate watersheds/outlets! For example, neosho has 3 separate watersheds (segment 13, 104, 186 as the outlet segments). 
* Calculate FSP and segment downstream distance (i.e., distance from outlet) using in interpolating FSP depth of water from gauges

At the end, this step updates the fsp_info and segment_info CSV files by adding additional columns. 

In [5]:
# calculate FSP and segment downstream distance for the tiled library
print(f'Calculate FSP and segment downstream distance for tiled library: {tiledLibFolder} ...')
CalculateFspSegmentDownstreamDistance(tiledLibFolder)

Calculate FSP and segment downstream distance for tiled library: E:/fldpln/workshop/devcon26/wildcat_10m_3dep/tiledlib ...


(      FspId          FspX          FspY  SegId  FilledElev   StrOrd  \
 0         1 -61658.205794  1.800128e+06      1  342.556946  1001003   
 1         2 -61648.205794  1.800118e+06      1  342.512115  1001003   
 2         3 -61648.205794  1.800108e+06      1  342.512115  1001003   
 3         4 -61638.205794  1.800098e+06      1  342.512115  1001003   
 4         5 -61628.205794  1.800088e+06      1  342.512115  1001003   
 ...     ...           ...           ...    ...         ...      ...   
 2653   2654 -48018.205794  1.792858e+06     16  302.513397  1001003   
 2654   2655 -48008.205794  1.792848e+06     16  302.507172  1001003   
 2655   2656 -47998.205794  1.792838e+06     16  302.501190  1001003   
 2656   2657 -47988.205794  1.792828e+06     16  302.495056  1001003   
 2657   2658 -47978.205794  1.792818e+06     16  302.488861  1001003   
 
             DsDist  
 0     31350.024510  
 1     31335.882374  
 2     31325.882374  
 3     31311.740239  
 4     31297.598103  
 .

## Move Stage-volume Tables to Tiled Library

In [3]:
# stage-volume table folder
svtFolder = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/src_stats' # folder containing the stage-volume tables
# tiledLibFolder = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/tiledlib' # tiled library folder where the stage-volume tables will be moved to

# Ensure the destination parent directory exists
if not os.path.exists(tiledLibFolder):
    os.makedirs(tiledLibFolder)

# Move the folder
try:
    new_path = shutil.move(svtFolder, tiledLibFolder)
    print(f"Stage-volume table folder successfully moved to: {new_path}")
except Exception as e:
    print(f"Error occurred: {e}")

Stage-volume table folder successfully moved to: E:/fldpln/workshop/devcon26/wildcat_10m_3dep/tiledlib\src_stats


## Assign Stream Orders to FSP and Segment CSV Files

This step gets stream orders from a segment shapefile, where each segment is assigned an order automatically or manually, and add the stream orders to their FSPs and segments in the fsp_info and segment_info CSV files. 

It also creates a new text file, stream_order_info.csv, which stores the connectivity among stream orders with columns: [‘StrOrd’, ‘DsStrOrd’, ‘JunctionFspX’, ‘JunctionFspY’]. This information is needed in horizontal and vertical interpolation methods. However, stream/segment orders are **NOT needed** using volume-based interpolation.

In [6]:
# Segment shapefile which has assigned stream orders. This shapefile is generated by the fldpln_model package.
orderShapefile = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/segs/wildcat_segments_ord4map.shp'
shpSegIdColName = 'segid' # or 'grid_code'
shpOrdColName = 'strord' # strord = 'subnetwork' * 1000 + 'snetstrord'

print(f'Get stream order and generate stream order network info for : {tiledLibFolder} ...')
GetStreamOrdersForFspsSegments(tiledLibFolder, orderShapefile, shpSegIdColName, shpOrdColName)

Get stream order and generate stream order network info for : E:/fldpln/workshop/devcon26/wildcat_10m_3dep/tiledlib ...


(      FspId          FspX          FspY  SegId  FilledElev        DsDist  \
 0         1 -61658.205794  1.800128e+06      1  342.556946  31350.024510   
 1         2 -61648.205794  1.800118e+06      1  342.512115  31335.882374   
 2         3 -61648.205794  1.800108e+06      1  342.512115  31325.882374   
 3         4 -61638.205794  1.800098e+06      1  342.512115  31311.740239   
 4         5 -61628.205794  1.800088e+06      1  342.512115  31297.598103   
 ...     ...           ...           ...    ...         ...           ...   
 2653   2654 -48018.205794  1.792858e+06     16  302.513397     56.568542   
 2654   2655 -48008.205794  1.792848e+06     16  302.507172     42.426407   
 2655   2656 -47998.205794  1.792838e+06     16  302.501190     28.284271   
 2656   2657 -47988.205794  1.792828e+06     16  302.495056     14.142136   
 2657   2658 -47978.205794  1.792818e+06     16  302.488861      0.000000   
 
        StrOrd  
 0     1001003  
 1     1001003  
 2     1001003  
 3    